# Esta parte do código se refere à pipeline da camada BRONZE em BATCH para testes antes de subir ao AWS

In [1]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# basedosdados se refere a base que o Governo Brasileiro disponíbiliza para análises
# pyarrow para salvar em PARQUET

!pip install basedosdados pyarrow --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import basedosdados as bd
import pandas as pd
import hashlib
import logging
from pathlib import Path

from datetime import datetime, timezone

In [3]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
PROJECT_ID = "tech-challenge-fase-2-502101"

DATASET = "br_inep_avaliacao_alfabetizacao"

TABELAS = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "alunos"
]

BUCKET = "fiap-alfabetizacao-ana-707472259268-us-east-1-an"

INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

CAMADA_BRONZE = "bronze"

In [4]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)

log = logging.getLogger(__name__)

In [5]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA BRONZE")
log.info(f"Projeto GCP : {PROJECT_ID}")
log.info(f"Dataset     : {DATASET}")
log.info(f"Bucket S3   : {BUCKET}")
log.info("~" * 35)

2026-08-18T22:07:51Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-18T22:07:51Z | INFO     | INICIANDO ETL DA CAMADA BRONZE
2026-08-18T22:07:51Z | INFO     | Projeto GCP : tech-challenge-fase-2-502101
2026-08-18T22:07:51Z | INFO     | Dataset     : br_inep_avaliacao_alfabetizacao
2026-08-18T22:07:51Z | INFO     | Bucket S3   : fiap-alfabetizacao-ana-707472259268-us-east-1-an
2026-08-18T22:07:51Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [6]:
QUERIES = {

    "uf": """
    WITH
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'uf'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'uf'
)
SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
    """,
    "meta_alfabetizacao_brasil": """ 
    SELECT
    dados.ano as ano,
    dados.rede as rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
    dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
    dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
    dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
    dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
    dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
    dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
    dados.percentual_participacao as percentual_participacao
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil` AS dados
    """,
     "meta_alfabetizacao_uf": """ 
    SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.rede as rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
    dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
    dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
    dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
    dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
    dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
    dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
    dados.percentual_participacao as percentual_participacao
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
    """,

    "meta_alfabetizacao_municipio": """ 
      SELECT
      dados.ano as ano,
      dados.id_municipio AS id_municipio,
      diretorio_id_municipio.nome AS id_municipio_nome,
      dados.rede as rede,
      dados.taxa_alfabetizacao as taxa_alfabetizacao,
      dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
      dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
      dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
      dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
      dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
      dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
      dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
      dados.nivel_alfabetizacao as nivel_alfabetizacao,
      dados.percentual_participacao as percentual_participacao
  FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio` AS dados
  LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
      ON dados.id_municipio = diretorio_id_municipio.id_municipio
    """,

    "municipio": """ 
    WITH 
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'municipio'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'municipio'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
     """,

    "alunos": """ 
    WITH 
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'alunos'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'alunos'
),
dicionario_presenca AS (
    SELECT
        chave AS chave_presenca,
        valor AS descricao_presenca
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'presenca'
        AND id_tabela = 'alunos'
),
dicionario_preenchimento_caderno AS (
    SELECT
        chave AS chave_preenchimento_caderno,
        valor AS descricao_preenchimento_caderno
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'preenchimento_caderno'
        AND id_tabela = 'alunos'
),
dicionario_alfabetizado AS (
    SELECT
        chave AS chave_alfabetizado,
        valor AS descricao_alfabetizado
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'alfabetizado'
        AND id_tabela = 'alunos'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.id_escola as id_escola,
    dados.id_aluno as id_aluno,
    dados.caderno as caderno,
    descricao_serie AS serie,
    descricao_rede AS rede,
    descricao_presenca AS presenca,
    descricao_preenchimento_caderno AS preenchimento_caderno,
    descricao_alfabetizado AS alfabetizado,
    dados.proficiencia as proficiencia,
    dados.peso_aluno as peso_aluno
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
LEFT JOIN `dicionario_presenca`
    ON dados.presenca = chave_presenca
LEFT JOIN `dicionario_preenchimento_caderno`
    ON dados.preenchimento_caderno = chave_preenchimento_caderno
LEFT JOIN `dicionario_alfabetizado`
    ON dados.alfabetizado = chave_alfabetizado
     """

}

In [7]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE LEITURA DAS QUERIES
"""
    Executa uma consulta SQL na Base dos Dados utilizando o BigQuery.

    Args:
        query (str): Consulta SQL a ser executada.

    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_base_dados(query):

    df = bd.read_sql(
        query=query,
        billing_project_id=PROJECT_ID
    )

    return df

In [8]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONSTRUINDO A CAMADA BRONZE
"""
    Adiciona metadados de ingestão ao DataFrame da camada Bronze.

    Args:
        df (pandas.DataFrame): Dados originais.
        dataset (str): Nome do dataset de origem.
        tabela (str): Nome da tabela de origem.
        
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def construir_bronze(df, dataset, tabela):

    log.info("Adicionando metadados da camada Bronze")

    df = df.copy()

    df["_ingestion_timestamp"] = INGESTION_TS
    df["_ingestion_date"] = INGESTION_DATE
    df["_source_dataset"] = dataset
    df["_source_table"] = tabela

    df["_record_hash"] = (
        df.astype(str)
          .apply(lambda row: hashlib.md5("".join(row).encode()).hexdigest(), axis=1)
    )

    log.info(f"{len(df)} registros preparados para camada Bronze")

    return df

In [9]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE QUALIDADE
# ~~~~~~~~~~~~~~~~~~~~~~~~

CHECKS = {  
    "uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_brasil": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "alunos": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_aluno", "critico": True},
        {"tipo": "unique", "coluna": "id_aluno", "critico": False},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
        {"tipo": "not_null", "coluna": "presenca", "critico": True},
    ]
}

In [10]:
# ~~~~~~~~~~~~~~
# DATA QUALITY
# ~~~~~~~~~~~~~~

def checar_qualidade(df, checks):
    """
    Executa as validações de qualidade da camada Bronze.

    Suporta os tipos: min_count, not_null e unique.
    Respeita o campo `critico`: se True (padrão), uma falha interrompe
    o pipeline (raise Exception); se False, a falha vira apenas um aviso
    no log (WARN) e a execução continua.

    Args:
        df (pandas.DataFrame): DataFrame da Bronze.
        checks (list): Lista de regras de validação.

    Raises:
        Exception: Caso alguma validação com critico=True falhe.
    """

    log.info("Iniciando verificações de qualidade")

    passou = 0
    falhou = 0

    for check in checks:

        tipo = check["tipo"]
        coluna = check.get("coluna")
        valor = check.get("valor")
        critico = check.get("critico", True)

        ok = False
        detalhe = ""

        if tipo == "min_count":

            quantidade = len(df)
            ok = quantidade >= valor
            detalhe = f"quantidade de registros={quantidade} | mínimo esperado={valor}"

        elif tipo == "not_null":

            nulos = df[coluna].isnull().sum()
            ok = nulos == 0
            detalhe = f"coluna '{coluna}' possui {nulos} valor(es) nulo(s)"

        elif tipo == "unique":

            duplicados = df.duplicated(subset=coluna).sum()
            ok = duplicados == 0
            detalhe = f"coluna(s) '{coluna}' possui(em) {duplicados} registro(s) duplicado(s)"

        else:

            log.warning(f"Tipo de check desconhecido, ignorado: '{tipo}'")
            continue

        status = "PASS" if ok else ("FAIL" if critico else "WARN")

        if ok:

            passou += 1
            log.info(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")

        else:

            falhou += 1

            if critico:
                log.error(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")
                raise Exception(f"Falha crítica de qualidade ({tipo}): {detalhe}")
            else:
                log.warning(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")

    log.info(f"Verificações de qualidade concluídas: {passou} passou(aram), {falhou} falhou(aram)")

In [11]:
# ~~~~~~~~~~~~~~~~~~~~
# SALVAR CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~

def salvar_bronze(df, tabela):

    # Particiona por ingestion_date (padrão Hive: chave=valor), para que
    # cada execução crie uma pasta nova em vez de sobrescrever a anterior.
    # Isso preserva o histórico completo de cargas, como o README promete,
    # e já deixa o layout pronto para um Glue Crawler detectar partições
    # automaticamente (particionamento físico em Parquet).
    pasta = Path("bronze") / tabela / f"ingestion_date={INGESTION_DATE}"
    pasta.mkdir(parents=True, exist_ok=True)

    arquivo = pasta / f"{tabela}.parquet"

    log.info(f"Salvando arquivo: {arquivo}")

    df.to_parquet(
        arquivo,
        index=False,
        engine="pyarrow"
    )

    log.info("Arquivo Parquet criado com sucesso.")

    return arquivo

In [12]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO PARA CAMADA BRONZE PARA REUTILIZAR EM VÁRIAS TABELAS
"""
    Executa o pipeline completo da camada Bronze para todas as tabelas
    configuradas no dicionário QUERIES.

    Etapas:
        1. Leitura da Base dos Dados.
        2. Construção da camada Bronze.
        3. Validação de qualidade dos dados.
        4. Geração do arquivo Parquet.
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def executar_bronze():
    for tabela, query in QUERIES.items():
      log.info("~" * 35)
      log.info(f"Iniciando execução da camada Bronze para '{tabela}'")
      log.info("~" * 35)

      # leitura da base dos dados
      df = ler_base_dados(query)

      # construindo a bronze com os metadados
      df_bronze = construir_bronze(df, DATASET, tabela)

      # exibe as primeiras linhas, somente usado no colab
      print(f"\nPrévia da tabela: {tabela}")
      display(df_bronze.head())

      # checks de integridades e tipos
      checks = CHECKS.get(tabela, [])

      # data quality
      if checks:
         checar_qualidade(df_bronze, checks)

      # salvando
      salvar_bronze(df_bronze, tabela)

log.info("Camada Bronze concluída com sucesso!")

2026-08-18T22:07:51Z | INFO     | Camada Bronze concluída com sucesso!


In [13]:
executar_bronze()

2026-08-18T22:07:51Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-18T22:07:51Z | INFO     | Iniciando execução da camada Bronze para 'uf'
2026-08-18T22:07:51Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Downloading: 100%|██████████|

2026-08-18T22:07:53Z | INFO     | Adicionando metadados da camada Bronze
2026-08-18T22:07:53Z | INFO     | 145 registros preparados para camada Bronze




Prévia da tabela: uf


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,...,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,uf,666f1f16a44a165b813a3dd43ff88f4c
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,uf,fbbbf62cf4ed7a278b1f003a4bb9b29d
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,uf,d680ab842a06ab13782a537162cf1aaf
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,uf,480dd5eec66a328dbb68fa11f69dab19
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,uf,590b791bcd71f0d357c2251edf3df203


...

Downloading: 100%|██████████|


2026-08-18T22:22:21Z | INFO     | Total time taken 848.22 s.
Finished at 2026-08-18 22:22:21.
2026-08-18T22:22:21Z | INFO     | Adicionando metadados da camada Bronze
2026-08-18T22:23:07Z | INFO     | 3867999 registros preparados para camada Bronze



Prévia da tabela: alunos


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,alunos,a93851ada570a60bb8a9203f986383bd
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,alunos,386f77f1374e6f2b48b32d6e4ca1efcf
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,alunos,f11e9a1a2cb05e43792e3390d28619f6
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,alunos,bb1b3f41df8f0be8c32b6e8e965cfa4f
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,20260819_010751,2026-08-19,br_inep_avaliacao_alfabetizacao,alunos,470a42c634920c0c97942580163f685e


2026-08-18T22:23:07Z | INFO     | Iniciando verificações de qualidade
2026-08-18T22:23:07Z | INFO     | [DQ:BRONZE] PASS | min_count | quantidade de registros=3867999 | mínimo esperado=1
2026-08-18T22:23:07Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'ano' possui 0 valor(es) nulo(s)
2026-08-18T22:23:08Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'id_aluno' possui 0 valor(es) nulo(s)
2026-08-18T22:23:09Z | WARNING  | [DQ:BRONZE] WARN | unique | coluna(s) 'id_aluno' possui(em) 1515671 registro(s) duplicado(s)
2026-08-18T22:23:09Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'id_escola' possui 0 valor(es) nulo(s)
2026-08-18T22:23:09Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'id_municipio' possui 0 valor(es) nulo(s)
2026-08-18T22:23:10Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'serie' possui 0 valor(es) nulo(s)
2026-08-18T22:23:10Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'rede' possui 0 valor(es) nulo(s)
2026-08-18T22:23:10Z | INFO     | [DQ:B